In [2]:
import sys
sys.path.append('..')  # so we can import from src/

import numpy as np
import pickle
import wfdb
from pathlib import Path

from src.harmonize.dataset import PhysioShiftDataset

ds = PhysioShiftDataset(target_fs=100.0, window_sec=10.0, overlap=0.5)

data_dir = Path.cwd().parent / 'data'

In [3]:
# --- D1: MIT-BIH (ECG, WFDB) ---
record = wfdb.rdrecord(str(data_dir / 'mitdb' / '100'))
signal = record.p_signal[:, 0]
ds.process_signal(signal, original_fs=record.fs, modality="ecg", domain_id="D1_MITBIH_open_chest")

# --- D2: PTB-XL (ECG, WFDB) ---
ptbxl_records = list((data_dir / 'ptbxl').rglob('*.dat'))
record = wfdb.rdrecord(str(ptbxl_records[0]).replace('.dat', ''))
signal = record.p_signal[:, 0]
ds.process_signal(signal, original_fs=record.fs, modality="ecg", domain_id="D2_PTBXL_open_clinical")

# --- D3: Multi-site PPG (PPG, WFDB) ---
multisite_records = list((data_dir / 'multisite_ppg').rglob('*.dat'))
record = wfdb.rdrecord(str(multisite_records[0]).replace('.dat', ''))
signal = record.p_signal[:, 0]
ds.process_signal(signal, original_fs=record.fs, modality="ppg", domain_id="D3_MultisitePPG_open_wearable")

# --- D4: WESAD (ECG, pickle) ---
with open(data_dir / 'wesad' / 'S2' / 'S2.pkl', 'rb') as f:
    wesad_data = pickle.load(f, encoding='latin1')
signal = np.asarray(wesad_data['signal']['chest']['ECG']).flatten()[:10000]
ds.process_signal(signal, original_fs=700.0, modality="ecg", domain_id="D4_WESAD_open_chest")

# --- D5: PPG-DaLiA (PPG, pickle) ---
with open(data_dir / 'ppgdalia' / 'PPG_FieldStudy' / 'S2' / 'S2.pkl', 'rb') as f:
    dalia_data = pickle.load(f, encoding='latin1')
signal = np.asarray(dalia_data['signal']['wrist']['BVP']).flatten()[:5000]
ds.process_signal(signal, original_fs=64.0, modality="ppg", domain_id="D5_PPGDaLiA_open_wrist")

ds.summary()


  PhysioShiftDataset cache summary
  D1_MITBIH_open_chest               : shape=(360, 1000)
  D2_PTBXL_open_clinical             : shape=(1, 1000)
  D3_MultisitePPG_open_wearable      : shape=(96, 1000)
  D4_WESAD_open_chest                : shape=(1, 1000)
  D5_PPGDaLiA_open_wrist             : shape=(14, 1000)


In [4]:
ds.save_cache('../data/harmonized_cache')

Saved D1_MITBIH_open_chest: shape=(360, 1000) -> ../data/harmonized_cache\D1_MITBIH_open_chest.npy
Saved D2_PTBXL_open_clinical: shape=(1, 1000) -> ../data/harmonized_cache\D2_PTBXL_open_clinical.npy
Saved D3_MultisitePPG_open_wearable: shape=(96, 1000) -> ../data/harmonized_cache\D3_MultisitePPG_open_wearable.npy
Saved D4_WESAD_open_chest: shape=(1, 1000) -> ../data/harmonized_cache\D4_WESAD_open_chest.npy
Saved D5_PPGDaLiA_open_wrist: shape=(14, 1000) -> ../data/harmonized_cache\D5_PPGDaLiA_open_wrist.npy
